# LeetCode 961 — N-Repeated Element in Size 2N Array bằng Python

**Link đề:** https://leetcode.com/problems/n-repeated-element-in-size-2n-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Pigeonhole Principle

```python
class Solution:
    def repeatedNTimes(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ đề để rút ra **tính chất đặc biệt**: chỉ có **một** giá trị xuất hiện nhiều hơn 1 lần.
2. Ôn `dict` đếm tần suất, `collections.Counter`, `set` để phát hiện phần tử trùng.
3. Hiểu **nguyên lý chuồng bồ câu** (pigeonhole) để có lời giải O(1) bộ nhớ: hai lần xuất hiện luôn cách nhau ≤ 3 vị trí.
4. Viết 5 cách: đếm brute force, Counter, set "gặp lại", sắp xếp, so khoảng cách ≤ 3.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Cho mảng `nums` có tính chất:
- `len(nums) == 2 * n`;
- `nums` có `n + 1` giá trị **khác nhau**, trong đó `n` giá trị xuất hiện **đúng một lần**;
- **đúng một** giá trị xuất hiện `n` lần.

Trả về giá trị xuất hiện `n` lần.

| nums | n | Kết quả |
|---|:---:|:---:|
| `[1,2,3,3]` | 2 | `3` |
| `[2,1,2,5,3,2]` | 3 | `2` |
| `[5,1,5,2,5,3,5,4]` | 4 | `5` |

**Constraints:** `2 <= n <= 5000`, `len(nums) == 2n`, `0 <= nums[i] <= 10^4`.

Đếm kiểm tra: `n` giá trị đơn lẻ + `n` bản sao của giá trị lặp = `2n` phần tử. Khớp!

In [1]:
examples = [([1, 2, 3, 3], 3), ([2, 1, 2, 5, 3, 2], 2), ([5, 1, 5, 2, 5, 3, 5, 4], 5)]
for nums, expected in examples:
    n = len(nums) // 2
    print(f"nums={nums}, n={n}, số giá trị khác nhau={len(set(nums))} (= n+1 = {n + 1}), đáp án={expected}")

nums=[1, 2, 3, 3], n=2, số giá trị khác nhau=3 (= n+1 = 3), đáp án=3
nums=[2, 1, 2, 5, 3, 2], n=3, số giá trị khác nhau=4 (= n+1 = 4), đáp án=2
nums=[5, 1, 5, 2, 5, 3, 5, 4], n=4, số giá trị khác nhau=5 (= n+1 = 5), đáp án=5


## 2. Nền tảng Python cần biết

### 2.1. `dict` đếm tần suất

Ý tưởng: `count[x]` = số lần `x` xuất hiện. Dùng `count.get(x, 0)` để lấy giá trị mặc định `0` khi `x` chưa có trong dict (tránh `KeyError`).

In [2]:
count = {}
for x in [2, 1, 2, 5, 3, 2]:
    count[x] = count.get(x, 0) + 1
print(count)
print("count.get(9, 0) =", count.get(9, 0))

{2: 3, 1: 1, 5: 1, 3: 1}
count.get(9, 0) = 0


### 2.2. `collections.Counter`

`Counter` là dict chuyên đếm, làm sẵn việc trên. `most_common(1)` trả về list một cặp `(giá_trị, số_lần)` lớn nhất.

In [3]:
from collections import Counter
c = Counter([5, 1, 5, 2, 5, 3, 5, 4])
print(c)
print(c.most_common(1))
print("giá trị:", c.most_common(1)[0][0])

Counter({5: 4, 1: 1, 2: 1, 3: 1, 4: 1})
[(5, 4)]
giá trị: 5


### 2.3. `set` — tập hợp, kiểm tra "đã gặp chưa?" O(1) trung bình

- Tạo: `seen = set()` (chú ý `{}` là dict rỗng, **không** phải set rỗng).
- Thêm: `seen.add(x)`. Kiểm tra: `x in seen`.

In [4]:
seen = set()
print(type({}), type(set()))
for x in [1, 2, 3, 3]:
    print(f"x={x}, đã gặp? {x in seen}")
    seen.add(x)

<class 'dict'> <class 'set'>
x=1, đã gặp? False
x=2, đã gặp? False
x=3, đã gặp? False
x=3, đã gặp? True


### 2.4. Nguyên lý chuồng bồ câu (Pigeonhole principle)

Nếu nhét **nhiều hơn** `k` con bồ câu vào `k` chuồng thì có ít nhất một chuồng chứa ≥ 2 con.

Áp dụng: chia mảng thành các "chuồng" là những đoạn liên tiếp. Nếu giá trị lặp có quá nhiều bản sao so với số đoạn, hai bản sao buộc phải rơi vào **cùng một đoạn ngắn** → chúng ở **gần nhau**. Ta sẽ dùng ý này ở mục 5.

In [5]:
# Ví dụ nhỏ: 5 chú chim, 4 chuồng -> chắc chắn có chuồng >= 2
pigeons = [0, 1, 2, 3, 0]          # chim i ở chuồng pigeons[i]
print(Counter(pigeons).most_common(1))

[(0, 2)]


## 3. Cách trực giác (baseline) — đếm từng phần tử

Với mỗi phần tử `x`, đếm số lần nó xuất hiện bằng `nums.count(x)`. Nếu bằng `n` thì đó là đáp án.

`list.count(x)` phải quét toàn bộ list → O(n) mỗi lần gọi.

In [6]:
def repeated_brute(nums: list[int]) -> int:
    n = len(nums) // 2
    for x in nums:
        if nums.count(x) == n:
            return x
    return -1

for nums, expected in examples:
    print(nums, "->", repeated_brute(nums), "| expected", expected)

[1, 2, 3, 3] -> 3 | expected 3
[2, 1, 2, 5, 3, 2] -> 2 | expected 2
[5, 1, 5, 2, 5, 3, 5, 4] -> 5 | expected 5


Độ phức tạp: mỗi phần tử gọi `count` O(2n) → tổng **O(n²)**. Với `n = 5000` là ~10⁸ phép so sánh trong trường hợp xấu — chậm.

## 4. Nút thắt và quan sát quan trọng

Nút thắt: ta **đếm lại** cùng một giá trị nhiều lần. Cách khắc phục chuẩn là đếm một lượt bằng dict/Counter → O(n) thời gian, O(n) bộ nhớ.

Nhưng hãy đọc lại đề: chỉ **một** giá trị xuất hiện nhiều hơn một lần, mọi giá trị khác xuất hiện **đúng một lần**.

➡ **Bất kỳ** giá trị nào ta gặp lần thứ hai **chắc chắn** là đáp án! Không cần đếm đến `n`.

In [7]:
def repeated_counter(nums: list[int]) -> int:
    return Counter(nums).most_common(1)[0][0]

def repeated_set(nums: list[int]) -> int:
    seen = set()
    for x in nums:
        if x in seen:          # gặp lại lần 2 -> chính là đáp án
            return x
        seen.add(x)
    return -1

for nums, expected in examples:
    print(nums, "->", repeated_counter(nums), repeated_set(nums), "| expected", expected)

[1, 2, 3, 3] -> 3 3 | expected 3
[2, 1, 2, 5, 3, 2] -> 2 2 | expected 2
[5, 1, 5, 2, 5, 3, 5, 4] -> 5 5 | expected 5


## 5. Insight O(1) bộ nhớ — hai bản sao luôn ở gần nhau

Câu hỏi: các bản sao của giá trị lặp có thể **cách xa nhau** đến mức nào?

Giả sử mọi cặp bản sao **liên tiếp** cách nhau ít nhất 3 vị trí (tức là `nums[i] != nums[i+1]` và `nums[i] != nums[i+2]` cho giá trị lặp). Khi đó `n` bản sao chiếm một đoạn dài ít nhất:

```text
1 + 3·(n − 1) = 3n − 2   vị trí
```

Đoạn này phải nằm gọn trong mảng dài `2n`, nên `3n − 2 ≤ 2n` ⇔ `n ≤ 2`.

- Nếu `n ≥ 3`: buộc phải có hai bản sao cách nhau **1 hoặc 2** vị trí.
- Nếu `n = 2` (mảng 4 phần tử): có thể xa nhất là `[x, a, b, x]` — cách nhau **3**.

➡ Chỉ cần kiểm tra `nums[i] == nums[i+k]` với `k ∈ {1, 2, 3}`. Bất kỳ cặp bằng nhau nào cũng chỉ có thể là giá trị lặp (vì các giá trị khác là duy nhất).

In [8]:
# Thử tìm một mảng n=3 mà các bản sao cách nhau >= 3: không thể
from itertools import permutations
n = 3
base = [9, 9, 9, 1, 2, 3]
far = [p for p in set(permutations(base))
       if all(p[i] != p[i + k] for k in (1, 2) for i in range(len(p) - k))]
print("Số hoán vị n=3 mà không có cặp cách 1 hoặc 2:", len(far))

n = 2
base = [9, 9, 1, 2]
far = [p for p in set(permutations(base))
       if all(p[i] != p[i + k] for k in (1, 2) for i in range(len(p) - k))]
print("n=2, các hoán vị như vậy:", far)

Số hoán vị n=3 mà không có cặp cách 1 hoặc 2: 0
n=2, các hoán vị như vậy: [(9, 2, 1, 9), (9, 1, 2, 9)]


Thí nghiệm khớp với chứng minh: với `n = 3` không có hoán vị nào, còn với `n = 2` chỉ có các dạng `[x, a, b, x]` — khoảng cách 3.

## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1:** với mỗi khoảng cách `k` từ 1 đến 3:
```python
for k in range(1, 4):
```
`range(1, 4)` sinh `1, 2, 3` (không gồm 4).

**Đoạn 2:** duyệt `i` sao cho `i + k` vẫn trong mảng: `range(len(nums) - k)`.

**Đoạn 3:** nếu `nums[i] == nums[i + k]` → trả về `nums[i]`.

In [9]:
nums = [1, 2, 3, 3]
for k in range(1, 4):
    print(f"k={k}: các cặp (i, i+k) =", [(i, i + k) for i in range(len(nums) - k)])

k=1: các cặp (i, i+k) = [(0, 1), (1, 2), (2, 3)]
k=2: các cặp (i, i+k) = [(0, 2), (1, 3)]
k=3: các cặp (i, i+k) = [(0, 3)]


In [10]:
def repeated_gap(nums: list[int]) -> int:
    for k in range(1, 4):                     # Đoạn 1
        for i in range(len(nums) - k):        # Đoạn 2
            if nums[i] == nums[i + k]:        # Đoạn 3
                return nums[i]
    return -1                                 # không xảy ra với input hợp lệ

for nums, expected in examples + [([9, 1, 2, 9], 9)]:
    print(nums, "->", repeated_gap(nums), "| expected", expected)

[1, 2, 3, 3] -> 3 | expected 3
[2, 1, 2, 5, 3, 2] -> 2 | expected 2
[5, 1, 5, 2, 5, 3, 5, 4] -> 5 | expected 5
[9, 1, 2, 9] -> 9 | expected 9


## 7. Trace từng bước

Ví dụ `nums = [9, 1, 2, 9]` (trường hợp xa nhất, `n = 2`):

| k | i | nums[i] | nums[i+k] | Bằng? |
|:---:|:---:|:---:|:---:|:---:|
| 1 | 0 | 9 | 1 | không |
| 1 | 1 | 1 | 2 | không |
| 1 | 2 | 2 | 9 | không |
| 2 | 0 | 9 | 2 | không |
| 2 | 1 | 1 | 9 | không |
| 3 | 0 | 9 | 9 | **có → trả 9** |

In [11]:
def trace_gap(nums: list[int]) -> int:
    print("nums =", nums)
    for k in range(1, 4):
        for i in range(len(nums) - k):
            same = nums[i] == nums[i + k]
            print(f"  k={k}, i={i}: nums[{i}]={nums[i]} vs nums[{i + k}]={nums[i + k]} -> {'BẰNG' if same else 'khác'}")
            if same:
                print("  => trả về", nums[i])
                return nums[i]
    return -1

trace_gap([9, 1, 2, 9])
print()
trace_gap([5, 1, 5, 2, 5, 3, 5, 4])

nums = [9, 1, 2, 9]
  k=1, i=0: nums[0]=9 vs nums[1]=1 -> khác
  k=1, i=1: nums[1]=1 vs nums[2]=2 -> khác
  k=1, i=2: nums[2]=2 vs nums[3]=9 -> khác
  k=2, i=0: nums[0]=9 vs nums[2]=2 -> khác
  k=2, i=1: nums[1]=1 vs nums[3]=9 -> khác
  k=3, i=0: nums[0]=9 vs nums[3]=9 -> BẰNG
  => trả về 9

nums = [5, 1, 5, 2, 5, 3, 5, 4]
  k=1, i=0: nums[0]=5 vs nums[1]=1 -> khác
  k=1, i=1: nums[1]=1 vs nums[2]=5 -> khác
  k=1, i=2: nums[2]=5 vs nums[3]=2 -> khác
  k=1, i=3: nums[3]=2 vs nums[4]=5 -> khác
  k=1, i=4: nums[4]=5 vs nums[5]=3 -> khác
  k=1, i=5: nums[5]=3 vs nums[6]=5 -> khác
  k=1, i=6: nums[6]=5 vs nums[7]=4 -> khác
  k=2, i=0: nums[0]=5 vs nums[2]=5 -> BẰNG
  => trả về 5


5

Ví dụ thứ hai dừng ngay ở `k = 2, i = 0` — với `n` lớn, ta thường tìm thấy rất sớm.

Trace thêm cho cách `set`:

In [12]:
def trace_set(nums: list[int]) -> int:
    seen = set()
    for i, x in enumerate(nums):
        print(f"i={i}, x={x}, seen trước={sorted(seen)}", end=" ")
        if x in seen:
            print("-> đã gặp, trả về", x)
            return x
        seen.add(x)
        print("-> thêm vào seen")
    return -1

trace_set([2, 1, 2, 5, 3, 2])

i=0, x=2, seen trước=[] -> thêm vào seen
i=1, x=1, seen trước=[2] -> thêm vào seen
i=2, x=2, seen trước=[1, 2] -> đã gặp, trả về 2


2

## 8. Code nộp LeetCode

### Cách A — set "gặp lại" (dễ nhớ, O(n) time, O(n) space)

In [13]:
class Solution:
    def repeatedNTimes(self, nums: list[int]) -> int:
        seen = set()
        for x in nums:
            if x in seen:
                return x
            seen.add(x)
        return -1

print(Solution().repeatedNTimes([1, 2, 3, 3]))
print(Solution().repeatedNTimes([2, 1, 2, 5, 3, 2]))
print(Solution().repeatedNTimes([5, 1, 5, 2, 5, 3, 5, 4]))

3
2
5


### Cách B — khoảng cách ≤ 3 (O(n) time, O(1) space)

In [14]:
class SolutionGap:
    def repeatedNTimes(self, nums: list[int]) -> int:
        for k in range(1, 4):
            for i in range(len(nums) - k):
                if nums[i] == nums[i + k]:
                    return nums[i]
        return -1

print(SolutionGap().repeatedNTimes([9, 1, 2, 9]))

9


### Cách C — sắp xếp

Sau khi sắp xếp, các bản sao đứng liền nhau. Với một giá trị chiếm **một nửa** mảng, vị trí giữa `n−1` hoặc `n` chắc chắn thuộc khối đó... nhưng cẩn thận: khối có thể nằm ở đầu (`[3,3,5,7]`) hoặc cuối (`[1,2,3,3]`). Cách an toàn: tìm cặp kề bằng nhau.

In [15]:
class SolutionSort:
    def repeatedNTimes(self, nums: list[int]) -> int:
        s = sorted(nums)
        for a, b in zip(s, s[1:]):
            if a == b:
                return a
        return -1

print(sorted([2, 1, 2, 5, 3, 2]), "->", SolutionSort().repeatedNTimes([2, 1, 2, 5, 3, 2]))

[1, 2, 2, 2, 3, 5] -> 2


### Cách D — Counter một dòng

In [16]:
class SolutionCounter:
    def repeatedNTimes(self, nums: list[int]) -> int:
        return Counter(nums).most_common(1)[0][0]

print(SolutionCounter().repeatedNTimes([5, 1, 5, 2, 5, 3, 5, 4]))

5


### Cách E (sáng tạo) — ngẫu nhiên

Chọn ngẫu nhiên hai chỉ số khác nhau `i, j`; nếu `nums[i] == nums[j]` thì đó là đáp án. Xác suất trúng mỗi lần ≈ `(n/2n)·((n−1)/(2n−1))` ≈ 1/4, nên kỳ vọng chỉ ~4 lần thử. Đây là ví dụ về **thuật toán Las Vegas**: luôn đúng, thời gian chạy ngẫu nhiên.

In [17]:
import random

class SolutionRandom:
    def repeatedNTimes(self, nums: list[int]) -> int:
        n2 = len(nums)
        while True:
            i, j = random.randrange(n2), random.randrange(n2)
            if i != j and nums[i] == nums[j]:
                return nums[i]

random.seed(0)
print(SolutionRandom().repeatedNTimes([5, 1, 5, 2, 5, 3, 5, 4]))

5


## 9. Vì sao đúng?

- **Cách set:** invariant — trước khi xét `nums[i]`, `seen` chứa đúng các giá trị ở `nums[0..i-1]`. Nếu `nums[i] in seen` thì giá trị này xuất hiện ≥ 2 lần. Theo đề, giá trị duy nhất như vậy là giá trị lặp → đúng. Vì giá trị lặp có `n ≥ 2` bản sao, lần thứ hai chắc chắn gặp trước khi hết mảng → luôn trả về.
- **Cách khoảng cách:** mục 5 đã chứng minh luôn tồn tại hai bản sao cách nhau 1, 2 hoặc 3. Ngược lại, mọi cặp `nums[i] == nums[i+k]` với `k ≥ 1` là hai vị trí khác nhau cùng giá trị → phải là giá trị lặp. Vậy giá trị trả về luôn đúng.
- **Cách sắp xếp:** sau sắp xếp, các bản sao liền kề; giá trị khác chỉ có 1 bản → cặp kề bằng nhau duy nhất là của giá trị lặp.

## 10. Độ phức tạp và so sánh

Gọi `m = 2n` là độ dài mảng.

| Cách | Time | Space | Nhận xét |
|---|---|---|---|
| Brute `count` | O(m²) | O(1) | dễ nghĩ, chậm |
| Counter | O(m) | O(m) | gọn, đếm toàn bộ |
| A. Set gặp lại | O(m) | O(m) | dừng sớm, dễ giải thích |
| C. Sắp xếp | O(m log m) | O(m) (sorted tạo bản sao) | không cần dict |
| **B. Khoảng cách ≤ 3** | **O(m)** (≤ 3 lượt) | **O(1)** | tận dụng pigeonhole |
| E. Ngẫu nhiên | O(1) kỳ vọng | O(1) | thú vị, không deterministic |

## 11. Bộ test

Gồm ví dụ đề, `n = 2` với khoảng cách 3, khối lặp ở đầu/cuối, giá trị 0, và test ngẫu nhiên sinh mảng hợp lệ để đối chiếu mọi cách.

In [18]:
tests = [
    ([1, 2, 3, 3], 3),
    ([2, 1, 2, 5, 3, 2], 2),
    ([5, 1, 5, 2, 5, 3, 5, 4], 5),
    ([9, 1, 2, 9], 9),          # n=2, khoảng cách 3
    ([0, 0, 1, 2], 0),          # giá trị 0, ở đầu
    ([1, 0, 2, 0], 0),          # khoảng cách 2
    ([7, 7, 7, 1, 2, 3], 7),    # khối ở đầu
    ([1, 2, 3, 7, 7, 7], 7),    # khối ở cuối
]
impls = {
    "brute": repeated_brute, "counter": repeated_counter, "set": repeated_set, "gap": repeated_gap,
    "LC-A": Solution().repeatedNTimes, "LC-B": SolutionGap().repeatedNTimes,
    "LC-C": SolutionSort().repeatedNTimes, "LC-D": SolutionCounter().repeatedNTimes,
    "LC-E": SolutionRandom().repeatedNTimes,
}
for nums, expected in tests:
    for name, f in impls.items():
        got = f(nums)
        assert got == expected, (name, nums, got, expected)
    print(f"{str(nums):28} -> {expected} PASS")
print("Tất cả test PASS!")

[1, 2, 3, 3]                 -> 3 PASS
[2, 1, 2, 5, 3, 2]           -> 2 PASS
[5, 1, 5, 2, 5, 3, 5, 4]     -> 5 PASS
[9, 1, 2, 9]                 -> 9 PASS
[0, 0, 1, 2]                 -> 0 PASS
[1, 0, 2, 0]                 -> 0 PASS
[7, 7, 7, 1, 2, 3]           -> 7 PASS
[1, 2, 3, 7, 7, 7]           -> 7 PASS
Tất cả test PASS!


In [19]:
def random_valid(rng, n):
    rep = rng.randint(0, 50)
    others = rng.sample([v for v in range(51) if v != rep], n)
    arr = [rep] * n + others
    rng.shuffle(arr)
    return arr, rep

rng = random.Random(961)
for _ in range(2000):
    arr, rep = random_valid(rng, rng.randint(2, 12))
    for name, f in impls.items():
        assert f(arr) == rep, (name, arr)
print("2000 mảng ngẫu nhiên hợp lệ: mọi cách đều đúng")

2000 mảng ngẫu nhiên hợp lệ: mọi cách đều đúng


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra khoảng cách 1 và 2

In [20]:
def wrong_gap_12(nums):
    for k in (1, 2):
        for i in range(len(nums) - k):
            if nums[i] == nums[i + k]:
                return nums[i]
    return -1

print("Sai :", wrong_gap_12([9, 1, 2, 9]))
print("Đúng:", repeated_gap([9, 1, 2, 9]))

Sai : -1
Đúng: 9


**Vì sao sai:** chứng minh ở mục 5 chỉ đảm bảo khoảng cách ≤ 2 khi `n ≥ 3`. Với `n = 2`, `[x, a, b, x]` có khoảng cách 3. **Sửa:** dùng `range(1, 4)`.

### Lỗi 2: Lấy phần tử giữa sau khi sắp xếp

In [21]:
def wrong_middle(nums):
    s = sorted(nums)
    return s[len(s) // 2]          # lấy s[n]

def wrong_middle_left(nums):
    s = sorted(nums)
    return s[len(s) // 2 - 1]      # lấy s[n-1]

print("s[n]   với [3,3,5,7]:", wrong_middle([3, 3, 5, 7]), "(đáp án 3) -> SAI")
print("s[n-1] với [1,2,3,3]:", wrong_middle_left([1, 2, 3, 3]), "(đáp án 3) -> SAI")

s[n]   với [3,3,5,7]: 5 (đáp án 3) -> SAI
s[n-1] với [1,2,3,3]: 2 (đáp án 3) -> SAI


**Vì sao sai:** khối `n` bản sao có thể nằm trọn nửa đầu (`[3,3,5,7]`, khối ở chỉ số 0..n-1) hoặc trọn nửa sau (`[1,2,3,3]`, chỉ số n..2n-1). Không có một vị trí cố định nào luôn thuộc khối. **Sửa:** tìm cặp kề bằng nhau trong mảng đã sắp xếp (Cách C).

### Lỗi 3: Dùng `{}` làm set rỗng

In [22]:
seen = {}
print(type(seen))      # dict, không phải set!
try:
    seen.add(1)
except AttributeError as e:
    print("Lỗi:", e)

<class 'dict'>
Lỗi: 'dict' object has no attribute 'add'


**Sửa:** `seen = set()`.

### Lỗi 4: Thêm vào `seen` trước khi kiểm tra

Nếu viết `seen.add(x)` rồi mới `if x in seen` thì điều kiện luôn đúng ngay phần tử đầu tiên → trả về `nums[0]` sai. Thứ tự đúng: **kiểm tra trước, thêm sau** (giống Two Sum).

### Lỗi 5: Kiểm tra `count == n` bằng `n = len(nums)`

Nhầm `n` với độ dài mảng. Ở đây `n = len(nums) // 2`.

## 13. Mẹo nhận dạng pattern

- Đề cho tính chất **"mọi giá trị khác xuất hiện đúng 1 lần"** → phần tử trùng đầu tiên chính là đáp án → `set` gặp lại.
- Đề cho một phần tử chiếm **tỉ lệ lớn** (≥ một nửa, ≥ 1/3...) → nghĩ đến **pigeonhole** (bản sao gần nhau), **Boyer–Moore voting** (nếu > một nửa), hoặc **ngẫu nhiên**.
- Muốn O(1) bộ nhớ mà không được sửa mảng → tìm **khoảng cách giới hạn** hoặc tính chất toán học thay vì cấu trúc dữ liệu.

## 14. Biến thể

1. **LeetCode 169 — Majority Element:** phần tử xuất hiện **> n/2** lần → Boyer–Moore voting O(1) bộ nhớ. (Bài 961 chỉ đúng **bằng** một nửa nên voting thuần không đảm bảo.)
2. **LeetCode 217 — Contains Duplicate:** có phần tử trùng không? → set.
3. **LeetCode 287 — Find the Duplicate Number:** giá trị trong `[1, n]`, không sửa mảng, O(1) bộ nhớ → Floyd tìm chu trình.
4. **LeetCode 229 — Majority Element II:** các phần tử xuất hiện > n/3 lần.

In [23]:
def majority_vote(nums):
    candidate, cnt = None, 0
    for x in nums:
        if cnt == 0:
            candidate = x
        cnt += 1 if x == candidate else -1
    return candidate

print("Majority (>n/2) của [2,2,1,1,1,2,2]:", majority_vote([2, 2, 1, 1, 1, 2, 2]))
print("Voting trên [1,2,3,3] (chỉ = n/2):", majority_vote([1, 2, 3, 3]), "-> may mắn đúng")
print("Voting trên [1,3,2,3] (chỉ = n/2):", majority_vote([1, 3, 2, 3]), "-> SAI, đáp án là 3")

Majority (>n/2) của [2,2,1,1,1,2,2]: 2
Voting trên [1,2,3,3] (chỉ = n/2): 3 -> may mắn đúng
Voting trên [1,3,2,3] (chỉ = n/2): 2 -> SAI, đáp án là 3


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Đếm tần suất bằng dict
Viết `freq(nums)` trả về dict giá trị → số lần (không dùng Counter).

In [24]:
def freq(nums: list[int]) -> dict:
    # TODO
    pass

# assert freq([2, 1, 2]) == {2: 2, 1: 1}

### Bài 2 — Phần tử trùng đầu tiên
Viết `first_duplicate(nums)` trả về giá trị đầu tiên **gặp lại** khi duyệt trái sang phải, hoặc `None`.

In [25]:
def first_duplicate(nums: list[int]):
    # TODO
    pass

# assert first_duplicate([3, 1, 4, 1, 3]) == 1
# assert first_duplicate([1, 2, 3]) is None

### Bài 3 — Khoảng cách nhỏ nhất giữa hai bản sao
Với mảng hợp lệ của bài 961, trả về khoảng cách chỉ số **nhỏ nhất** giữa hai bản sao của giá trị lặp.

In [26]:
def min_gap(nums: list[int]) -> int:
    # TODO
    pass

# assert min_gap([9, 1, 2, 9]) == 3
# assert min_gap([5, 1, 5, 2, 5, 3, 5, 4]) == 2
# assert min_gap([1, 2, 3, 3]) == 1

### Bài 4 — Kiểm tra mảng hợp lệ
Viết `is_valid_2n(nums)` kiểm tra mảng có đúng tính chất đề bài không (độ dài chẵn, n+1 giá trị khác nhau, một giá trị xuất hiện n lần).

In [27]:
def is_valid_2n(nums: list[int]) -> bool:
    # TODO
    pass

# assert is_valid_2n([1, 2, 3, 3]) == True
# assert is_valid_2n([1, 1, 2, 2]) == False
# assert is_valid_2n([1, 2, 3]) == False

### Bài 5 — Mọi phần tử xuất hiện nhiều hơn một lần
Trả về list các giá trị xuất hiện ≥ 2 lần, theo thứ tự lần **đầu tiên** xuất hiện.

In [28]:
def all_duplicates(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [3, 2]

### Bài 6 (nâng cao) — Majority element (> n/2) với O(1) bộ nhớ
Viết `majority(nums)` bằng Boyer–Moore voting, giả sử chắc chắn tồn tại phần tử xuất hiện > `len(nums)/2` lần.

In [29]:
def majority(nums: list[int]) -> int:
    # TODO
    pass

# assert majority([2, 2, 1, 1, 1, 2, 2]) == 2
# assert majority([3, 3, 4]) == 3

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | một lượt duyệt | `d.get(x, 0)` | `d[x] = d.get(x, 0) + 1` |
| 2 | nhớ cái đã thấy | `set` | kiểm tra `in` trước rồi `add` |
| 3 | lưu chỉ số lần cuối gặp | dict giá trị → chỉ số | `best = min(best, i - last[x])` |
| 4 | dùng Counter | số khóa phải là `n+1` | `max(c.values()) == n` |
| 5 | đếm + thứ tự | dict giữ thứ tự chèn | lọc `c[x] >= 2` theo thứ tự key |
| 6 | ứng viên + bộ đếm | gặp cùng thì +1, khác thì −1 | bộ đếm về 0 → đổi ứng viên |

## 17. Đáp án tham khảo

In [30]:
def ans_freq(nums):
    d = {}
    for x in nums:
        d[x] = d.get(x, 0) + 1
    return d

def ans_first_duplicate(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return x
        seen.add(x)
    return None

def ans_min_gap(nums):
    last, best = {}, len(nums)
    for i, x in enumerate(nums):
        if x in last:
            best = min(best, i - last[x])
        last[x] = i
    return best

def ans_is_valid_2n(nums):
    if len(nums) % 2:
        return False
    n = len(nums) // 2
    c = Counter(nums)
    return len(c) == n + 1 and max(c.values()) == n and sorted(c.values()) == [1] * n + [n]

def ans_all_duplicates(nums):
    c = Counter(nums)
    return [x for x in c if c[x] >= 2]

def ans_majority(nums):
    cand, cnt = None, 0
    for x in nums:
        if cnt == 0:
            cand = x
        cnt += 1 if x == cand else -1
    return cand

assert ans_freq([2, 1, 2]) == {2: 2, 1: 1}
assert ans_first_duplicate([3, 1, 4, 1, 3]) == 1
assert ans_first_duplicate([1, 2, 3]) is None
assert ans_min_gap([9, 1, 2, 9]) == 3
assert ans_min_gap([5, 1, 5, 2, 5, 3, 5, 4]) == 2
assert ans_min_gap([1, 2, 3, 3]) == 1
assert ans_is_valid_2n([1, 2, 3, 3]) == True
assert ans_is_valid_2n([1, 1, 2, 2]) == False
assert ans_is_valid_2n([1, 2, 3]) == False
assert ans_all_duplicates([4, 3, 2, 7, 8, 2, 3, 1]) == [3, 2]
assert ans_majority([2, 2, 1, 1, 1, 2, 2]) == 2
assert ans_majority([3, 3, 4]) == 3
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def repeatedNTimes(self, nums: list[int]) -> int:
        # O(1) bộ nhớ: hai bản sao luôn cách nhau 1, 2 hoặc 3
        for k in range(1, 4):
            for i in range(len(nums) - k):
                if nums[i] == nums[i + k]:
                    return nums[i]
        return -1
```

Hoặc bản dễ nhớ: `seen = set()`; gặp `x in seen` thì trả `x`.

**Checklist trước khi submit**
- [ ] Đã tận dụng "chỉ một giá trị bị lặp" → phần tử trùng đầu tiên là đáp án?
- [ ] Nếu dùng khoảng cách: đã gồm `k = 3` cho trường hợp `[x, a, b, x]`?
- [ ] Set rỗng viết `set()`, không phải `{}`?
- [ ] Kiểm tra `in seen` **trước** khi `add`?
- [ ] Không lấy "phần tử giữa" sau khi sắp xếp?